# 09.08 - Self-supervised encoder linear probing

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Linear-probe versus fine-tuning comparison.

Treat a prepared encoder as the output of self-supervised pretraining, freeze it for a linear probe, then fine-tune a copied encoder under the same labeled split.

## Core Ideas

A linear probe measures information already present in frozen representations. Full fine-tuning changes the representation and usually costs more. Prevent label leakage by fitting both strategies only on training labels and evaluating the same untouched validation observations.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

SEED = 9
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared Labeled Features and Encoder

Sixty observations form three classes. The first 45 train observations and final 15 validation observations are disjoint. The fixed encoder represents a locally available SSL checkpoint.

**Model return structure — `prepared_encoder`:** Maps float tensors `[N,6]` to embeddings `[N,4]` on the same device.

In [ ]:
labels = np.repeat(np.arange(3), 20)
raw_features = rng.normal(0.0, 0.12, size=(60, 6)).astype(np.float32)
raw_features[:, :3] += np.eye(3, dtype=np.float32)[labels]
train_indices = np.concatenate([np.arange(c * 20, c * 20 + 15) for c in range(3)])
validation_indices = np.concatenate([np.arange(c * 20 + 15, c * 20 + 20) for c in range(3)])
train_features, validation_features = torch.from_numpy(raw_features[train_indices]), torch.from_numpy(raw_features[validation_indices])
train_labels, validation_labels = torch.from_numpy(labels[train_indices]).long(), torch.from_numpy(labels[validation_indices]).long()
prepared_encoder = nn.Linear(6, 4, bias=False)
with torch.no_grad():
    prepared_encoder.weight.copy_(torch.tensor([[1.,0.,0.,0.,0.,0.],[0.,1.,0.,0.,0.,0.],[0.,0.,1.,0.,0.,0.],[0.5,0.5,0.5,0.,0.,0.]]))
print("split/support:", len(train_indices), len(validation_indices), torch.bincount(validation_labels).tolist())

## Exercise 09-A: Extract frozen embeddings

Use evaluation mode and inference mode; never retain a graph for a frozen probe.

**Return structure — `extract_embeddings`:** A detached CPU float32 tensor `[N,4]` for input `[N,6]`.

In [ ]:
def extract_embeddings(encoder, features, device=DEVICE):
    encoder = encoder.to(device).eval()
    with torch.inference_mode():
        return encoder(features.to(device)).detach().cpu().to(torch.float32)


# Smoke check: extract both disjoint splits.
train_embeddings = extract_embeddings(prepared_encoder, train_features)
validation_embeddings = extract_embeddings(prepared_encoder, validation_features)
print(train_embeddings.shape, validation_embeddings.shape)

## Exercise 09-B: Fit a linear probe

Fit only a linear classifier on frozen training embeddings and report validation Macro-F1.

**Return structure — `run_linear_probe`:** A dictionary with `predictions` as an integer NumPy array `[N_val]`, Python floats `macro_f1` and `runtime_seconds`, and `validation_support` as `list[int]`.

In [ ]:
def run_linear_probe(train_embeddings, train_targets, validation_embeddings, validation_targets):
    start = time.perf_counter()
    classifier = LogisticRegression(max_iter=200, random_state=SEED)
    classifier.fit(train_embeddings.numpy(), train_targets.numpy())
    predictions = classifier.predict(validation_embeddings.numpy()).astype(np.int64)
    targets = validation_targets.numpy()
    return {"predictions": predictions, "macro_f1": float(f1_score(targets, predictions, average="macro")), "runtime_seconds": time.perf_counter() - start, "validation_support": np.bincount(targets, minlength=3).astype(int).tolist()}


# Smoke check: evaluate frozen features.
linear_probe_result = run_linear_probe(train_embeddings, train_labels, validation_embeddings, validation_labels)
print("linear probe:", linear_probe_result)

## Exercise 09-C: Fine-tune an encoder copy

Copy the prepared linear encoder, attach a classifier, and update both only with training labels.

**Return structure — `run_fine_tuning`:** A dictionary with `predictions` as an integer NumPy array `[N_val]`, Python floats `macro_f1` and `runtime_seconds`, and `validation_support` as `list[int]`.

In [ ]:
def run_fine_tuning(encoder, train_data, train_targets, validation_data, validation_targets, epochs=20, device=DEVICE):
    tuned_encoder = nn.Linear(encoder.in_features, encoder.out_features, bias=encoder.bias is not None)
    tuned_encoder.load_state_dict({key: value.detach().cpu().clone() for key, value in encoder.state_dict().items()})
    model = nn.Sequential(tuned_encoder, nn.ReLU(), nn.Linear(encoder.out_features, 3)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    start = time.perf_counter()
    for _ in range(epochs):
        optimizer.zero_grad(); loss = nn.functional.cross_entropy(model(train_data.to(device)), train_targets.to(device)); loss.backward(); optimizer.step()
    model.eval()
    with torch.inference_mode():
        predictions = model(validation_data.to(device)).argmax(dim=1).cpu().numpy().astype(np.int64)
    targets = validation_targets.numpy()
    return {"predictions": predictions, "macro_f1": float(f1_score(targets, predictions, average="macro")), "runtime_seconds": time.perf_counter() - start, "validation_support": np.bincount(targets, minlength=3).astype(int).tolist()}


# Smoke check: run the aligned fine-tuning comparison.
fine_tune_result = run_fine_tuning(prepared_encoder, train_features, train_labels, validation_features, validation_labels)
print("fine-tune:", fine_tune_result)

## Exercise 09-D: Compare evidence

Expose the same validation support, Macro-F1, runtime, and delta from the frozen baseline.

**Return structure — `probe_comparison_table`:** A two-row DataFrame with columns `strategy`, `train_size`, `validation_size`, `validation_support`, `macro_f1`, `runtime_seconds`, and `delta_macro_f1`.

In [ ]:
def probe_comparison_table(linear_result, fine_tune_result):
    rows = [{"strategy": "linear_probe", "train_size": len(train_labels), "validation_size": len(validation_labels), **{key: linear_result[key] for key in ["validation_support", "macro_f1", "runtime_seconds"]}}, {"strategy": "fine_tune", "train_size": len(train_labels), "validation_size": len(validation_labels), **{key: fine_tune_result[key] for key in ["validation_support", "macro_f1", "runtime_seconds"]}}]
    table = pd.DataFrame(rows)
    table["delta_macro_f1"] = table["macro_f1"] - float(table.iloc[0]["macro_f1"])
    return table


# Smoke check: print the controlled comparison.
probe_evidence = probe_comparison_table(linear_probe_result, fine_tune_result)
print(probe_evidence.to_string(index=False))

## Test Cases

**Return structure — `run_day09_tests`:** Returns `None`; assertions and `Day 09 tests passed` communicate success.

In [ ]:
def run_day09_tests():
    assert set(train_indices).isdisjoint(set(validation_indices))
    assert train_embeddings.shape == (45, 4) and validation_embeddings.shape == (15, 4)
    assert not train_embeddings.requires_grad
    assert linear_probe_result["predictions"].shape == (15,)
    assert fine_tune_result["predictions"].shape == (15,)
    assert linear_probe_result["validation_support"] == fine_tune_result["validation_support"] == [5, 5, 5]
    assert probe_evidence.shape == (2, 7) and float(probe_evidence.iloc[0]["delta_macro_f1"]) == 0.0
    print("Day 09 tests passed")


run_day09_tests()

## Day 09 Checklist

- [ ] Extract frozen embeddings without gradients.
- [ ] Fit the probe only on training labels.
- [ ] Fine-tune a copied encoder on the same split.
- [ ] Compare Macro-F1 and runtime on all validation rows.
- [ ] Run the test cases.